## Setup

In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade -q botocore boto3

Note: you may need to restart the kernel to use updated packages.


In [3]:
import boto3

In [4]:
sess = boto3.Session(profile_name='')
bedrock = sess.client('bedrock', 'us-west-2')
response = bedrock.list_foundation_models()
print(response.get('modelSummaries')) 

[{'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-tg1-large', 'modelId': 'amazon.titan-tg1-large', 'modelName': 'Titan Text Large', 'providerName': 'Amazon', 'inputModalities': ['TEXT'], 'outputModalities': ['TEXT'], 'responseStreamingSupported': True, 'customizationsSupported': ['FINE_TUNING'], 'inferenceTypesSupported': ['ON_DEMAND']}, {'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-e1t-medium', 'modelId': 'amazon.titan-e1t-medium', 'modelName': 'Titan Text Embeddings', 'providerName': 'Amazon', 'inputModalities': ['TEXT'], 'outputModalities': ['EMBEDDING'], 'customizationsSupported': [], 'inferenceTypesSupported': ['ON_DEMAND']}, {'modelArn': 'arn:aws:bedrock:us-west-2::foundation-model/amazon.titan-embed-g1-text-02', 'modelId': 'amazon.titan-embed-g1-text-02', 'modelName': 'Titan Text Embeddings v2', 'providerName': 'Amazon', 'inputModalities': ['TEXT'], 'outputModalities': ['EMBEDDING'], 'customizationsSupported': [], 'inferenceTypesSup

In [5]:
import boto3

In [6]:
bedrock = sess.client('bedrock-runtime', 'us-west-2')

In [7]:
def initialise_bedrock():
    global bedrock
    if bedrock is None:
        sess = boto3.Session(profile_name='')
        bedrock = sess.client('bedrock-runtime', 'us-west-2')

## Get products into dynamodb

In [27]:
!aws s3 cp --profile  s3://YOUR_BUCKET_NAME/FlipkartProd/flipkart_com-ecommerce_sample.csv .

download: s3://YOUR_BUCKET_NAME/FlipkartProd/flipkart_com-ecommerce_sample.csv to ./flipkart_com-ecommerce_sample.csv


In [8]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [9]:
import csv
import json
from decimal import Decimal
from tqdm import tqdm
import boto3
from boto3.dynamodb.types import TypeSerializer

In [10]:
#Initialize the DynamoDB resource
dynamodb = sess.resource('dynamodb')

In [31]:
def is_json(value):
    try:
        json.loads(value)
        return True
    except:
        return False

In [32]:
def extract_lists(row, column):
    list_string = row.get(column, "").strip('[]').strip('"')
    if column == "product_category_tree":
        return [cat.strip() for cat in list_string.split(">>")]
    else:
        return [url.strip() for url in list_string.split(",")]

In [33]:
def simplify_product_specification(product_spec_json):
    simplified_json = {}
    for item in product_spec_json.get('product_specification', []):
        key = item.get('key', None)
        value = item.get('value', None)
        if key and value:
            simplified_json[key] = value
    return simplified_json

In [29]:


serializer = TypeSerializer()

def convert_to_dynamodb_format_enhanced(value, column=None):
    if column == "product_specifications":
        try:
            # Replace "=>" with ":" to make it a valid Python object
            corrected_value = value.replace("=>", ":")
            # Use ast.literal_eval to convert the string to a Python object
            python_object = json.loads(corrected_value)
            # Serialize the Python object to DynamoDB format
            return serializer.serialize(simplify_product_specification(python_object))
        except Exception as e:
            # If conversion fails, store it as a string
            return {"S": value.strip('"')}
    if column and 'price' in column:
        # Convert the price to a float
        if value.strip():
            return serializer.serialize(Decimal(str(value)))
        else:
            return {"NULL": True}
    else:
        # Serialize other columns using the TypeSerializer
        return serializer.serialize(value.strip('"'))

In [30]:
# Function to assign parent IDs to the category items in the adjacency list
def assign_parent_ids(adjacency_list_items):
    parent_child_dict = {}
    
    for item in adjacency_list_items:
        category_id = item['PutRequest']['Item']['category_id']['S']
        parent_id = item['PutRequest']['Item']['parent_id']['S']
        
        if parent_id == "TBD":
            parent_id = parent_child_dict.get(category_id, "null")
            item['PutRequest']['Item']['parent_id']['S'] = parent_id
            
# Updated function to convert CSV to DynamoDB compatible JSON using Adjacency List for category trees
def csv_to_dynamodb_json_with_adjacency_list(csv_file_path, json_file_path):
    parent_child_dict = {}  # To store parent-child relationships
    dynamodb_items = []  # For storing product items
    adjacency_list_items = []  # For storing category items
    
    with open(csv_file_path, 'r') as csv_file:
        csv_reader = csv.DictReader(csv_file)
        
        for row in csv_reader:
            dynamodb_item = {}
            categories = extract_lists(row, "product_category_tree")
            images = extract_lists(row, "image")
            
            # Update parent-child relationships
            for i, category in enumerate(categories):
                parent = categories[i - 1] if i > 0 else "null"
                parent_child_dict[category] = parent
            
            for column, value in row.items():
                if column == "product_category_tree":
                    dynamodb_item[column] = {"L": [convert_to_dynamodb_format_enhanced(cat) 
                                                   for cat in categories]}
                elif column == "image":
                    dynamodb_item[column] = {"L": [convert_to_dynamodb_format_enhanced(img) for img in images]}
                elif column == "uniq_id":
                    dynamodb_item['id'] = convert_to_dynamodb_format_enhanced(value, column)
                else:
                    dynamodb_item[column] = convert_to_dynamodb_format_enhanced(value, column)
                    
            dynamodb_items.append({"PutRequest": {"Item": dynamodb_item}})
    
    # Generate the adjacency list items
    for category, parent in parent_child_dict.items():
        adjacency_list_items.append({
            "PutRequest": {
                "Item": {
                    "category_id": {"S": category},
                    "parent_id": {"S": parent}
                }
            }
        })
    
    # Assign parent IDs
    assign_parent_ids(adjacency_list_items)
    
    with open(json_file_path, 'w') as json_file:
        json.dump({
            "ProductTable": {"Items": dynamodb_items},
            "CategoryTable": {"Items": adjacency_list_items}
        }, json_file, indent=4)

In [31]:
csv_to_dynamodb_json_with_adjacency_list("flipkart_com-ecommerce_sample.csv", "flipkart_com-ecommerce_sample.json")

In [32]:
 

# Create 'ProductTable'
product_table = dynamodb.create_table(
    TableName='ProductTable',
    KeySchema=[
        {'AttributeName': 'id', 'KeyType': 'HASH'}
    ],
    AttributeDefinitions=[
        {'AttributeName': 'id', 'AttributeType': 'S'}
    ],
    ProvisionedThroughput={
        'ReadCapacityUnits': 25,
        'WriteCapacityUnits': 25
    }
)

# Create 'CategoryTable'
category_table = dynamodb.create_table(
    TableName='CategoryTable',
    KeySchema=[
        {'AttributeName': 'category_id', 'KeyType': 'HASH'}
    ],
    AttributeDefinitions=[
        {'AttributeName': 'category_id', 'AttributeType': 'S'}
    ],
    ProvisionedThroughput={
        'ReadCapacityUnits': 25,
        'WriteCapacityUnits': 25
    }
)

# Wait for tables to be created
product_table.meta.client.get_waiter('table_exists').wait(TableName='ProductTable')
category_table.meta.client.get_waiter('table_exists').wait(TableName='CategoryTable')


In [33]:

def batch_write_chunks(table_name, table_data):
    dynamodb = sess.client('dynamodb')
    chunk_size = 25  # DynamoDB batch write limit
    
    # Extract the list of items wrapped in 'PutRequest'
    items_list = table_data.get('Items', [])
    
    for i in tqdm(range(0, 2000 if table_name=='ProductTable' else (int(len(items_list))), chunk_size)):
        chunk = items_list[i:i + chunk_size]
        request = {
            table_name: chunk
        }
        dynamodb.batch_write_item(RequestItems=request)

# Load JSON data from the file
with open('flipkart_com-ecommerce_sample.json', 'r') as f:
    data = json.load(f)

# Batch write items to 'ProductTable'
if 'ProductTable' in data:
    batch_write_chunks('ProductTable', data['ProductTable'])

# Batch write items to 'CategoryTable'
if 'CategoryTable' in data:
    batch_write_chunks('CategoryTable', data['CategoryTable'])


100%|██████████| 362/362 [01:14<00:00,  4.88it/s]


## Get records

## Set up product desc generation

In [8]:
dynamodb = sess.resource('dynamodb')
table = dynamodb.Table('ProductTable')

lastEvaluatedKey = None
items = [] # Result Array

while True:
    if lastEvaluatedKey == None:
        response = table.scan() 
    else:
        response = table.scan(
        ExclusiveStartKey=lastEvaluatedKey
    )

    items.extend(response['Items'])

    if 'LastEvaluatedKey' in response:
        lastEvaluatedKey = response['LastEvaluatedKey']
    else:
        break

print(len(items))

2000


In [9]:
from decimal import Decimal
from typing import Union, Dict, Any

def json_to_dict(json_input: Union[Dict[str, Any], str]) -> Dict[str, Any]:
    if isinstance(json_input, dict):
        return json_input
    elif isinstance(json_input, str):
        try:
            return json.loads(json_input.replace("=>", ":"))
        except json.JSONDecodeError:
            return {}
    else:
        return {}

def format_decimal(decimal_value: Union[Decimal, str, int]) -> str:
    return str(decimal_value)

def array_to_string(array: list) -> str:
    return ' > '.join(array)

def generate_prompt(product_data: Dict[str, Any]) -> str:
    description = product_data.get('description', '')
    discounted_price = format_decimal(product_data.get('discounted_price', ''))
    retail_price = format_decimal(product_data.get('retail_price', ''))
    product_name = product_data.get('product_name', '')
    product_category_tree = array_to_string(product_data.get('product_category_tree', []))
    product_specifications = json_to_dict(product_data.get('product_specifications', {}))

    template = (
        f"Given the following product details:\n"
        f"- Product ID: {product_data.get('id', '')}\n"
        f"- Brand: {product_data.get('brand', '')}\n"
        f"- Original Description: {description}\n"
        f"- Discounted Price: {discounted_price}\n"
        f"- Retail Price: {retail_price}\n"
        f"- Product Name: {product_name}\n"
        f"- Product Category: {product_category_tree}\n"
        f"- Product Specifications: {product_specifications.get('product_specification', '')}\n\n"
        f"Please generate an enhanced product description that incorporates all the above elements while ensuring "
        f"high-quality language and factual coherence. Make the description rich in relevant details "
        f"and present it in a format that includes:\n"
        f"- A compelling opening sentence\n"
        f"- Key features\n"
        f"- Benefits\n"
        f"- Usage instructions\n"
        f"- Pricing information\n\n"
    )
    return template


test = generate_prompt(items[13]) 


In [10]:
%pip install backoff

Note: you may need to restart the kernel to use updated packages.


In [11]:
import backoff

def backoff_hdlr(details):
    """Handler from https://pypi.org/project/backoff/"""
    '''print(
        "Backing off {wait:0.1f} seconds after {tries} tries "
        "calling function {target} with kwargs "
        "{kwargs}".format(**details)
    )'''

In [12]:
from botocore.exceptions import ClientError
import random
import json

@backoff.on_exception(
        backoff.expo,
        (ClientError),
        max_time=random.randint(500, 1000),
        on_backoff=backoff_hdlr,
        giveup=lambda e: 'ThrottlingException' not in str(e) and 'ReadTimeoutError' not in str(e)
    ) 
def generate_description(product_data: dict) -> str:
    generated_description = product_data.get('generated_description')
    if generated_description and 'Sorry' not in generated_description and len(generated_description) > 0:
        return generated_description
    prompt = generate_prompt(product_data)
    claude_prompt = f"\n\nHuman:{prompt}\n\nAssistant:"
    body = json.dumps({"prompt": claude_prompt,"max_tokens_to_sample":2000,
                       "temperature":0,"top_p":1,"stop_sequences": ["\n\nHuman:"],"top_k": 250})
    modelId = 'anthropic.claude-v2'
    accept = 'application/json'
    contentType = 'application/json'
    response = bedrock.invoke_model(body=body, modelId=modelId, accept=accept,
                                    contentType=contentType)
    response_body = json.loads(response.get('body').read())
    if 'Sorry' in response_body.get('completion'):
        print(f'No description found for product:{product_data["description"]}')
        return ''
    #remove first line from response using \n
    return ' '.join(response_body.get('completion').split('\n',2)[2:])

In [13]:
generate_prompt(items[200])

"Given the following product details:\n- Product ID: 65f186769e0fa2a60adb8db67297d8f5\n- Brand: Pruto\n- Original Description: Key Features of Pruto Lace Up Ocassion :Formal Closure Type :Lace-Up Sole : PU Outer Material :Faux Leather Please refer to Size-Chart for proper fit,Pruto Lace Up Price: Rs. 1,740 Add a bold flavour to your casual look by wearing these Formals from Pruto. These lace-up shoe features a fine design which will accentuate your look. The polyUrethane sole further ensures comfort throughout the day.,Specifications of Pruto Lace Up General Occasion Formal Ideal For Men Shoe Details Sole Material Rubber Number of Contents in Sales Package Pack of 1 Weight 477 g (per single Shoe) - Weight of the product may vary depending on size. Heel Height 0.5 inch Outer Material Artificial Leather Color Black In the Box 1 Pair of Shoes\n- Discounted Price: 1740\n- Retail Price: 1740\n- Product Name: Pruto Lace Up\n- Product Category: Footwear > Men's Footwear > Formal Shoes > Pruto

In [14]:
generate_description(items[900])

"Make a stylish statement with the Claude Lorrain Black Casuals Shoes. Crafted for the modern woman, these shoes are designed to add flair to any outfit. The product features a sleek black colorway with intricate stitching details, cushioned insoles for all-day comfort, and durable rubber outsoles to keep you steady on your feet. \n\nShowcase your personal style with these versatile shoes that seamlessly transition from work to weekend. The medium heel provides just the right amount of lift whether you're headed to the office or out on the town. Claude Lorrain is known for premium quality and these shoes deliver with a luxe leather upper and high-grade materials. \n\nGet the designer look without the designer price tag. Valued at Rs. 2750, these Claude Lorrain shoes are now available for just Rs. 1299 exclusively on Flipkart. Take advantage of this limited-time deal to add these must-have casuals to your wardrobe. \n\nSlip into effortless sophistication with the Claude Lorrain Black Ca

In [15]:
import random
print(generate_description(items[random.randint(0,len(items)-1)]))

Make a bold fashion statement with the Kielz Ladies Boots. Crafted for style and comfort, these stylish boots are a must-have addition to your wardrobe. The opening sentence highlights the key benefit - style and comfort.

Featuring a sleek and modern design, these ankle-high boots are designed to keep you looking chic all day long. The round toe silhouette and low block heel create a flattering and feminine look, while the premium synthetic leather upper delivers durability. The key features bring out the product's design, materials and silhouette. 

Slip your feet into these boots and experience all-day comfort. The lightly padded insole and textile lining ensure a soft feel against your skin, while the durable rubber outsole provides traction and stability with every step. The benefits focus on the comfort aspects that make this an appealing everyday boot.

Simply pair these versatile boots with your favorite jeans, leggings or dresses for an effortlessly cool vibe. The low heel mak

## Generate Descriptions and write to dynamo

In [16]:
from tqdm import tqdm

In [17]:
from botocore.exceptions import ClientError
from tqdm import tqdm


def update_dynamodb(product_id, generated_description):
    try:
        response = table.update_item(
            Key={
                'id': product_id,
            },
            UpdateExpression="set generated_description = :d",
            ExpressionAttributeValues={
                ':d': generated_description,
            },
            ReturnValues="UPDATED_NEW"
        )
    except ClientError as e:
        print(e.response['Error']['Message'])
    else:
        return response

def update_generated_descriptions(items):
    for item in tqdm(items):
        new_description = generate_description(item)
        update_dynamodb(item['id'], new_description)

In [18]:
def split_data(data, num_splits):
    avg = len(data) // num_splits
    out = []
    last = 0.0

    while last < len(data):
        out.append(data[int(last):int(last + avg)])
        last += avg

    return out


In [19]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import sys

def process_chunk_of_items(items_chunk):
    
    initialise_bedrock()
    
    update_generated_descriptions(items_chunk)


NUM_THREADS = 400  
items_chunks = split_data(items, NUM_THREADS)


with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = [executor.submit(process_chunk_of_items, chunk) for chunk in tqdm(items_chunks)]


  0%|          | 0/400 [00:00<?, ?it/s]










































































































































































 75%|███████▌  | 301/400 [00:00<00:00, 2993.87it/s]


















100%|██████████| 400/400 [00:00<00:00, 896.33it/s] 
























































































































































































100%|██████████| 5/5 [00:00<00:00,  6.96it/s]

































100%|██████████| 5/5 [00:01<00:00,  4.12it/s]

100%|██████████| 5/5 [00:01<00:00,  4.00it/s]



100%|██████████| 5/5 [00:01<00:00,  3.98it/s]


























100%|██████████| 5/5 [00:01<00:00,  3.75it/s]



































100%|██████████| 5/5 [00:01<00:00,  3.62it/s]










100%|██████████| 5/5 [00:01<00:00,  3.55it/s]
























































## Image Captioning

## Clean UP

In [ ]:
%cd ~/vs

In [ ]:
!rm -r code
!rm -r $model_id

In [55]:
def delete_table(table_name):
    table = dynamodb.Table(table_name)
    table.delete()

In [56]:
!rm flipkart_com-ecommerce_sample.json
!rm flipkart_com-ecommerce_sample.csv

In [57]:
'''delete_table('ProductTable')
delete_table('CategoryTable')'''